**Bài làm:** Nguyễn Bá Chính — 2A202602654 · K4-Track02-Day18

Đường lightweight; code gốc và assertion được giữ. Phân tích cuối notebook có hỗ trợ AI; số liệu lấy từ kernel thực thi. Xem `../AI_USAGE.md`.

In [1]:
import sys
from pathlib import Path
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "scripts" / "lakehouse.py").exists())
sys.path.insert(0, str(repo_root / "notebooks"))
sys.path.insert(0, str(repo_root / "scripts"))


# NB1 — Delta Lake Basics (lightweight path)

**Stack:** `deltalake` (delta-rs) + Polars + DuckDB. No Spark, no JVM.
Maps to slide §2 (Delta Lake) + deliverable bullet 1.

> Spark equivalent: `spark.read.format("delta").load(path)` ↔ `DeltaTable(path).to_pyarrow_table()`.
> Same on-disk format, different binding.

In [2]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path (file-relative)
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "users_delta")
reset(table_path)  # idempotent rerun

## 1. Write a Delta table

In [3]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "age": [30, 25, 35],
    "city": ["Hanoi", "HCMC", "Danang"],
})
write_deltalake(table_path, df.to_arrow(), mode="overwrite")

## 2. Read it back + inspect transaction log

Look at `_lakehouse/scratch/users_delta/_delta_log/00000000000000000000.json` —
that's the transaction log. Same JSON format Spark/Databricks would write.

In [4]:
dt = DeltaTable(table_path)
print(pl.from_arrow(dt.to_pyarrow_table()))
print("\nHistory:")
for h in dt.history():
    print(f"  v{h['version']}  {h['operation']}  {h.get('operationMetrics', {})}")

shape: (3, 4)
┌─────┬─────────┬─────┬────────┐
│ id  ┆ name    ┆ age ┆ city   │
│ --- ┆ ---     ┆ --- ┆ ---    │
│ i64 ┆ str     ┆ i64 ┆ str    │
╞═════╪═════════╪═════╪════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   │
│ 3   ┆ charlie ┆ 35  ┆ Danang │
└─────┴─────────┴─────┴────────┘

History:
  v0  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 4, 'num_retries': 0}


## 3. Schema enforcement — try to write a wrong schema

In [5]:
bad = pl.DataFrame({"id": [4], "name": ["dan"], "age": ["thirty"], "city": ["Hue"]})
schema_blocked = False
version_before_bad_write = DeltaTable(table_path).version()
try:
    write_deltalake(table_path, bad.to_arrow(), mode="append")
    print("UNEXPECTED: bad write succeeded — schema enforcement broken")
except Exception as e:
    schema_blocked = True
    msg = str(e).splitlines()[0][:120]
    print(f"BLOCKED by schema enforcement (expected): {type(e).__name__}: {msg}")
assert schema_blocked, "Schema enforcement must reject age='thirty'"
assert DeltaTable(table_path).version() == version_before_bad_write, "Rejected write changed the table"
assert DeltaTable(table_path).count() == 3, "Rejected write added rows"

BLOCKED by schema enforcement (expected): Exception: Cast error: Cannot cast string 'thirty' to value of Int64 type


## 4. Schema evolution (opt-in)

In [6]:
new = pl.DataFrame({
    "id": [4], "name": ["dan"], "age": [28], "city": ["Hue"], "tier": ["premium"],
})
write_deltalake(table_path, new.to_arrow(), mode="append", schema_mode="merge")
dt = DeltaTable(table_path)
# Sort by id so the printout is stable across reruns — Delta does not
# preserve write-order across appends.
print(pl.from_arrow(dt.to_pyarrow_table()).sort("id"))

shape: (4, 5)
┌─────┬─────────┬─────┬────────┬─────────┐
│ id  ┆ name    ┆ age ┆ city   ┆ tier    │
│ --- ┆ ---     ┆ --- ┆ ---    ┆ ---     │
│ i64 ┆ str     ┆ i64 ┆ str    ┆ str     │
╞═════╪═════════╪═════╪════════╪═════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  ┆ null    │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   ┆ null    │
│ 3   ┆ charlie ┆ 35  ┆ Danang ┆ null    │
│ 4   ┆ dan     ┆ 28  ┆ Hue    ┆ premium │
└─────┴─────────┴─────┴────────┴─────────┘


## 5. Query with DuckDB via Arrow (part of the required notebook)

In [7]:
import duckdb

# We hand DuckDB an Arrow table rather than calling `delta_scan()`. delta_scan
# autoloads a DuckDB extension over the network — fine at home, a support
# ticket in a firewalled classroom. Arrow registration is zero-copy and offline.
con = duckdb.connect()
con.register("users", DeltaTable(table_path).to_pyarrow_table())
tier_counts = con.sql("SELECT tier, count(*) AS n FROM users GROUP BY 1 ORDER BY 1").fetchall()
print(tier_counts)

[('premium', 1), (None, 3)]


## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] schema_mode="merge" added the `tier` column
- [ ] DuckDB query returned 2 tier groups
The flag below records the actual failed write; the error is preserved above.

In [8]:
from pathlib import Path as _Path  # noqa: E402

_log = sorted(_Path(table_path).glob("_delta_log/*.json"))
print("Transaction log files:", [f.name for f in _log])
print("First commit JSON (actual file content):")
print(_log[0].read_text(encoding="utf-8"))
_cols = DeltaTable(table_path).schema().to_arrow().names
checks = {
    "_delta_log/ has JSON commits": len(_log) >= 2,
    "schema enforcement blocked bad write": schema_blocked,
    "tier column added via schema_mode=merge": "tier" in _cols,
    "duckdb sees 2 tier groups": len(tier_counts) == 2,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB1 incomplete — see FAIL rows above"
print("\nNB1 complete.")

Transaction log files: ['00000000000000000000.json', '00000000000000000001.json']
First commit JSON (actual file content):
{"commitInfo":{"timestamp":1791104949741,"operation":"WRITE","operationParameters":{"mode":"Overwrite"},"engineInfo":"delta-rs:py-1.6.6","operationMetrics":{"num_added_files":1,"num_removed_files":0,"num_partitions":0,"num_added_rows":3,"execution_time_ms":4,"num_retries":0},"clientVersion":"delta-rs.py-1.6.6"}}
{"protocol":{"minReaderVersion":1,"minWriterVersion":2}}
{"metaData":{"id":"b73f159d-deb8-40cc-b0c6-f28fbccd2076","name":null,"description":null,"format":{"provider":"parquet","options":{}},"schemaString":"{\"type\":\"struct\",\"fields\":[{\"name\":\"id\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"name\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"age\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"city\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}}]}","partitionColumns":[],"createdTi

## Số liệu kiểm chứng của bài nộp

In [9]:
import json as submission_json
submission_metrics = dict(log_files=[p.name for p in _log], schema_blocked=schema_blocked,
             table_version=DeltaTable(table_path).version(), rows=DeltaTable(table_path).count(),
             columns=_cols, tier_groups=tier_counts)
print('SUBMISSION_METRICS_JSON=' + submission_json.dumps(submission_metrics, ensure_ascii=False, default=str))

SUBMISSION_METRICS_JSON={"log_files": ["00000000000000000000.json", "00000000000000000001.json"], "schema_blocked": true, "table_version": 1, "rows": 4, "columns": ["id", "name", "age", "city", "tier"], "tier_groups": [["premium", 1], [null, 3]]}


## Giải thích kết quả

Có 2 commit JSON trong `_delta_log/`. Ghi `age='thirty'` bị chặn; kiểm tra bổ sung xác nhận version và số dòng không đổi sau lần ghi lỗi. Opt-in `schema_mode='merge'` thêm `tier`, bảng hiện có 4 dòng; ba dòng cũ mang NULL, dòng mới mang premium. Log ghi metadata/schema và hành động add file của commit; một thư mục Parquet đơn thuần không có transaction log này.